In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB",
    )

CUDA available: True
GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
from google.colab import userdata
userdata.get('tooken')

In [15]:
import pandas as pd

df = pd.read_csv("test-2.csv")

print(df.shape)
print(df.columns)

(460, 12)
Index(['filename', 'start', 'end', 'content', 'techniques',
       'persuasion_strategies', 'Attack_on_Reputation', 'Justification',
       'Distraction', 'Simplification', 'Call', 'Manipulative_Wording'],
      dtype='object')


In [16]:
first_100 = df.head(100).copy()

first_100.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,pl_statements_20_02_2025_n03.txt,3935,3957,Wyjątkowa organizacja.,[],[],False,False,False,False,False,False
1,pl_hate_speech_act_19_12_2024_n10.txt,5526,5636,"To dobrze, że proponujemy uwzględnić kolejne p...",[],[],False,False,False,False,False,False
2,pl_defence_07_03_2025_n08.txt,1032,1196,Wojsko tak naprawdę nie jest zbyt skomplikowan...,['Exaggeration-Minimisation'],['Manipulative_Wording'],False,False,False,False,False,True
3,pl_abortion_11_04_2024_n01.txt,4565,4579,My się cofamy.,['Repetition'],['Manipulative_Wording'],False,False,False,False,False,True
4,pl_defence_22_05_2024_n03.txt,818,920,"Warto o tym w tej chwili wspomnieć, dlatego że...",['Loaded_Language'],['Manipulative_Wording'],False,False,False,False,False,True


In [17]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from transformers import BitsAndBytesConfig

model_id = "lapa-llm/lapa-v0.1.3-instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [18]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


In [19]:
import bitsandbytes as bnb
print(bnb.__version__)

0.50.2


In [22]:
HF_TOKEN = "tokeeeen"
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
).eval()

processor = AutoProcessor.from_pretrained(
    model_id,
    token=HF_TOKEN,
)

config.json:   0%|          | 0.00/3.12k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 37.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

### Test

In [23]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "Translate the following Polish text into Ukrainian.\n"
                    "Return only the Ukrainian translation.\n\n"
                    "Rzeczywiście to jest hańba."
                ),
            }
        ],
    }
]

In [24]:
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

In [25]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
)

translation = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(translation)

Дійсно, це ганьба.


### Translation

In [26]:
def translate_text(text: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Translate the following Polish text into Ukrainian.\n"
                        "Return only the Ukrainian translation.\n"
                        "Preserve the meaning, names, numbers, quotations and tone.\n\n"
                        f"Polish text:\n{text}"
                    ),
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

    result = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return result.strip()

In [27]:
import os

OUTPUT_FILE = "test-2.csv"

first_100["content_uk"] = ""

for i in range(100):

    if first_100.loc[i, "content_uk"] != "":
        continue

    print(f"Translating {i + 1}/100...")

    translation = translate_text(
        str(first_100.loc[i, "content"])
    )

    first_100.loc[i, "content_uk"] = translation

    first_100.to_csv(OUTPUT_FILE, index=False)

    print(translation)
    print("-" * 80)

Translating 1/100...
Виняткова організація.
--------------------------------------------------------------------------------
Translating 2/100...
Добре, що пропонуємо врахувати наступні фактори: вік, стать, інвалідність та сексуальну орієнтацію.
--------------------------------------------------------------------------------
Translating 3/100...
Армія насправді не є надто складною інституцією, бо там йдеться про те, щоб виграти, а не програти, і щоб мати чим виграти, а не щоб мати чим програти.
--------------------------------------------------------------------------------
Translating 4/100...
Ми відступаємо.
--------------------------------------------------------------------------------
Translating 5/100...
Варто про це згадати зараз, тому що сьогодні ситуація дійшла до певного критичного моменту.
--------------------------------------------------------------------------------
Translating 6/100...
Голова БНБ сказав, що у нас є 3 роки, щоб підготуватися до можливої конфронтації.
----

KeyboardInterrupt: 

###### All train data

In [29]:
import os

OUTPUT_FILE = "PL_test_translated_lapa.csv"

# Create target column if it doesn't exist
if "content_uk" not in df.columns:
    df["content_uk"] = ""

for i in range(len(df)):
    # Resume safely if disconnected
    if df.loc[i, "content_uk"] != "":
        continue

    print(f"Translating {i + 1}/{len(df)}...")

    translation = translate_text(str(df.loc[i, "content"]))
    df.loc[i, "content_uk"] = translation

    # Save progress after each sentence
    df.to_csv(OUTPUT_FILE, index=False)

    print(translation)
    print("-" * 80)

##### All train data using batching

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [31]:
import os

OUTPUT_FILE = "/content/drive/MyDrive/PL_train_translated.csv"

# Resume if a previous output exists
if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE, keep_default_na=False)
else:
    df = pd.read_csv("test-2.csv")
    df["content_uk"] = ""

processor.tokenizer.padding_side = "left"

def build_messages(text):
    return [{
        "role": "user",
        "content": [{"type": "text", "text": (
            "Translate the following Polish text into Ukrainian.\n"
            "Return only the Ukrainian translation.\n"
            "Preserve the meaning, names, numbers, quotations and tone.\n\n"
            f"Polish text:\n{text}"
        )}],
    }]

@torch.inference_mode()
def translate_batch(texts):
    inputs = processor.apply_chat_template(
        [build_messages(t) for t in texts],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": True, "padding_side": "left"},
    ).to(model.device)

    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[-1]:]
    return [t.strip() for t in processor.batch_decode(gen, skip_special_tokens=True)]

# Only untranslated rows, shortest first
todo = df[df["content_uk"].astype(str).str.strip() == ""].copy()
todo["len"] = todo["content"].astype(str).str.len()
todo = todo.sort_values("len")
idx_list = todo.index.tolist()

BATCH = 8   # raise to 12-16 if memory allows; lower if you get OOM
for b in range(0, len(idx_list), BATCH):
    idx = idx_list[b:b+BATCH]
    texts = df.loc[idx, "content"].astype(str).tolist()
    try:
        df.loc[idx, "content_uk"] = translate_batch(texts)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        for i in idx:  # fall back to one at a time
            df.loc[i, "content_uk"] = translate_batch([str(df.loc[i, "content"])])[0]

    if (b // BATCH) % 5 == 0:   # checkpoint every ~40 sentences
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"{(df['content_uk'].astype(str).str.strip() != '').sum()}/{len(df)} done")

df.to_csv(OUTPUT_FILE, index=False)

KeyboardInterrupt: 